In [ ]:
import numpy as np
import random

Define the SimpleEnvironment class

In [ ]:
class SimpleEnvironment:
    def __init__(self):
        self.states = [0, 1, 2, 3, 4]  # 5 states
        self.goal_state = 4  # Goal is state 4
        self.current_state = 0  # Agent starts at state 0

    def reset(self):
        self.current_state = 0
        print("\nEnvironment reset. Starting at state 0.")
        return self.current_state

    def step(self, action):
        # Action: 0 = left, 1 = right
        direction = "left" if action == 0 else "right"
        prev_state = self.current_state

        if action == 0:
            self.current_state = max(0, self.current_state - 1)
        elif action == 1:
            self.current_state = min(4, self.current_state + 1)

        reward = 1 if self.current_state == self.goal_state else 0
        done = self.current_state == self.goal_state
        print(f"Action taken: {direction} | State: {prev_state} → {self.current_state} | Reward: {reward}")
        return self.current_state, reward, done


Define the ActorCriticAgent class

In [ ]:
class ActorCriticAgent:
    def __init__(self, num_states, num_actions, alpha_actor=0.1, alpha_critic=0.1, gamma=0.9, serotonin_level=0.5):
        self.num_states = num_states
        self.num_actions = num_actions
        self.gamma = gamma
        self.alpha_actor = alpha_actor
        self.alpha_critic = alpha_critic

        # Initialise the critic (state-value function) and the actor (policy)
        self.V = np.zeros(num_states)  # State value function V(s)
        self.policy = np.ones((num_states, num_actions)) / num_actions  # Action probabilities π(a|s)

        # Serotonin influences exploration, modulating the softmax temperature
        self.temperature = 1.0 - serotonin_level  # Higher serotonin -> less exploration
        print(f"Agent initialised with serotonin level: {serotonin_level} → Temperature: {self.temperature:.2f}")

    def softmax(self, preferences):
        scaled_prefs = preferences / self.temperature
        exps = np.exp(scaled_prefs - np.max(scaled_prefs))  # Stability trick
        return exps / np.sum(exps)

    def choose_action(self, state):
        probs = self.softmax(self.policy[state])
        action = np.random.choice(self.num_actions, p=probs)
        print(f"Choosing action at state {state} with policy probs {probs.round(2)} → Chosen action: {action}")
        return action

    def update(self, state, action, reward, next_state):
        td_error = reward + self.gamma * self.V[next_state] - self.V[state]
        print(f"TD Error (δ): {td_error:.4f} | V[{state}] before update: {self.V[state]:.4f}")

        # Critic update
        self.V[state] += self.alpha_critic * td_error
        print(f"Updated V[{state}]: {self.V[state]:.4f}")

        # Actor update
        probs = self.softmax(self.policy[state])
        for a in range(self.num_actions):
            if a == action:
                self.policy[state][a] += self.alpha_actor * td_error * (1 - probs[a])
            else:
                self.policy[state][a] -= self.alpha_actor * td_error * probs[a]
        print(f"Updated policy at state {state}: {self.policy[state].round(2)}")

Initialise the environment and agent

In [ ]:
env = SimpleEnvironment()
agent = ActorCriticAgent(num_states=5, num_actions=2, serotonin_level=0.3)

Train the agent over multiple episodes

In [ ]:
num_episodes = 100
for episode in range(num_episodes):
    print(f"\n=== Episode {episode + 1} ===")
    state = env.reset()
    total_reward = 0

    for step in range(20):  # Limit the number of steps in an episode
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.update(state, action, reward, next_state)
        state = next_state
        total_reward += reward

        if done:
            print("Goal state reached!")
            break

    print(f"Episode {episode + 1} finished. Total Reward = {total_reward}")
